# 3 · Baseline, transects and rate statistics


## 3.1 Baseline and transects

Following the conventions of the USGS Digital Shoreline Analysis System (DSAS v5.0, Himmelstoss et al. 2018 [8]), we first build a reference **baseline** landward of all shorelines, and then cast **transects** perpendicular to it at fixed spacing. Here the baseline is the *median* shoreline position across all eleven scenes — a choice that is robust against the seasonal wiggle of the beach and against any single-scene outlier — smoothed twice with a Savitzky–Golay filter and offset 100 m landward so every transect begins in clear land. Transects are cast at **500 m spacing** along the 110.6 km baseline (222 transects in total), extending 600 m seaward.

For every transect–shoreline pair we record the distance from the transect's land end to the shoreline, measured along the transect. Because distances grow towards the sea, **larger values mean accretion and smaller values mean erosion**.

## 3.2 Rate statistics

Three DSAS-style statistics are computed per transect:

| Statistic | Definition | Use |
|---|---|---|
| **EPR** — End Point Rate | Net displacement between the earliest and latest shoreline divided by elapsed years | Robust, needs only two shorelines |
| **LRR** — Linear Regression Rate | Slope of the OLS regression of shoreline position versus decimal date, with its standard error (LSE) and 90 % confidence interval (LCE) | Recommended DSAS default; uses all shorelines |
| **WLR** — Weighted Linear Regression | Same regression, weighted by the inverse of scene cloud cover | Emphasises the clearest acquisitions |

All computations are in metres because the working CRS is UTM zone 31N (EPSG:32631). The full pipeline and the three publication figures are produced by the script shown below.


```bash
cd .. && PYTHONPATH=. python3 scripts/04_baseline_transects.py
cd .. && PYTHONPATH=. python3 scripts/05_compute_rates.py
cd .. && PYTHONPATH=. python3 scripts/06_make_figures.py
```


In [ ]:
import numpy as np
import pandas as pd
from scipy import stats as sp_stats

dist = pd.read_csv("data/shoreline_distances.csv")

# decimal years since the earliest shoreline
def to_years(series: pd.Series):
    s = series.dropna().sort_index()
    return (s.index - s.index[0]).total_seconds() / 31557600.0

def epr(series: pd.Series):
    s = series.dropna().sort_index()
    years = to_years(s)
    return (s.iloc[-1] - s.iloc[0]) / years[-1]

def lrr(series: pd.Series):
    # slope, standard error, R^2 of position ~ time
    s = series.dropna().sort_index()
    x, y = to_years(s).values, s.values
    slope, _, r, _, se = sp_stats.linregress(x, y)
    return slope, se, r ** 2

# example: transect 100 (the strongest erosion hotspot)
t100 = dist[dist.transect_id == 100].set_index("date")["distance_m"]
print(f"net change : {t100.iloc[-1] - t100.iloc[0]:+.1f} m")
slope, se, r2 = lrr(t100)
print(f"LRR        : {slope:.2f} ± {1.645 * se:.2f} m/yr "
      f"(90% CI, R² = {r2:.2f})")
print(f"EPR        : {epr(t100):.2f} m/yr")

